In [15]:
import pandas as pd
import pickle

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


In [16]:
with open("../artifacts/tunedXgboost.pkl", "rb") as f:
    data = pickle.load(f)

model = data["model"]
threshold = data["threshold"]


In [17]:
df=pd.read_csv('../data/loanDefaulter.csv')

In [18]:
df['LoanPurpose'] = df['LoanPurpose'].map({
    'Home': 0,
    'Auto': 1,
    'Education': 1,
    'Other': 2,
    'Business': 3
})

df['MaritalStatus'] = df['MaritalStatus'].map({
    "Single": 0,
    "Married": 1,
    "Divorced": 0
})

df['EmploymentType'] = df['EmploymentType'].map({
    'Full-time': 0,
    'Self-employed': 1,
    'Part-time': 2,
    'Unemployed': 3
})

df['Education'] = df['Education'].map({
    "High School": 1,
    "Bachelor's": 2,
    "Master's": 3,
    "PhD": 4
})


In [19]:
X = df.drop(columns=["Default", "LoanID"], axis=1)
y = df["Default"]

In [20]:
manual_encoded_cols = ['LoanPurpose', 'MaritalStatus', 'EmploymentType', 'Education']

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

categorical_cols = [col for col in categorical_cols if col not in manual_encoded_cols]

numerical_cols = [col for col in X.columns if col not in categorical_cols]


In [21]:
preprocessor = ColumnTransformer([
    ("num", "passthrough", numerical_cols),
    ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

In [22]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

In [23]:
print("Training pipeline...")
pipeline.fit(X, y)
print("Training complete!")

Training pipeline...


c:\Users\SOHAM\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [17:07:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training complete!


In [24]:
with open("../artifacts/completeXgboost.pkl", "wb") as f:
    pickle.dump({
        "pipeline": pipeline,
        "threshold": threshold
    }, f)

print("Pipeline saved!")

Pipeline saved!


In [25]:


sample = X.iloc[[0]]

y_prob = pipeline.predict_proba(sample)[:, 1]
y_pred = (y_prob > threshold).astype(int)

print("\nSample Prediction:")
print("Probability:", y_prob[0])
print("Prediction:", "Default" if y_pred[0] == 1 else "No Default")


Sample Prediction:
Probability: 0.18504164
Prediction: No Default
